# AI Praktik Tapsiriq - CyberShield
logistic regression ile riskli girisleri tapmaq (0 = normal, 1 = riskli)

## 1. melumati taniyaq

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, precision_score, recall_score, f1_score)

In [ ]:
df = pd.read_csv("CyberShield_LogisticRegression_Dataset.csv", encoding="utf-8-sig")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

bosluqlar bir nece sutunda var, onlari asagida train uzerinde median ile dolduracam. login_id sadece koddur deye modele vermirem.

In [ ]:
print(df["risky_login"].value_counts())
print(df["risky_login"].value_counts(normalize=True))

df["risky_login"].value_counts().plot(kind="bar", color=["green", "red"])
plt.title("risky_login siniflari")
plt.xlabel("sinif (0 normal, 1 riskli)")
plt.ylabel("say")
plt.xticks(rotation=0)
plt.show()

siniflerin sayi eyni deyil, ona gore bolguden sonra stratify istifade edecem ki train ve testde pay eyni qalsin.

## 2. hazirliq

In [ ]:
X = df.drop(columns=["login_id", "risky_login"])
y = df["risky_login"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

imputer ve scaler yalniz train-e fit olunur, test ucun sadece transform. yoxsa testden melumat sizar (data leakage).

In [ ]:
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_imp)
X_test_s = scaler.transform(X_test_imp)

print(np.isnan(X_train_s).sum(), np.isnan(X_test_s).sum())

## 3. model

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:, 1]

print(y_pred[:10])
print(y_proba[:10].round(3))

## 4. sehvleri arasdiraq

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print(cm)
print("TN", tn, "FP", fp, "FN", fn, "TP", tp)

print("accuracy :", round(accuracy_score(y_test, y_pred), 3))
print("precision:", round(precision_score(y_test, y_pred), 3))
print("recall   :", round(recall_score(y_test, y_pred), 3))
print("f1       :", round(f1_score(y_test, y_pred), 3))

- **FP (false positive):** giris normaldi, model riskli dedi. istifadeci narahat olur, elave yoxlama gedir, amma hesab tehlukesizdir.
- **FN (false negative):** giris real riskli idi, model normal dedi. hucum kecib gede biler, buna gore bu daha tehlukelidir.

tehlukesizlikde FN daha pisdir, cunki tutulmayan riskli giris hesabin ele kecirilmesi demekdir. FP ise sadece vaxt itkisidir. ona gore recall bizim ucun precision-dan vacibdir.

## 5. qrafikler

In [ ]:
ConfusionMatrixDisplay(cm, display_labels=["normal", "riskli"]).plot(cmap="Blues")
plt.title("Confusion Matrix (hed 0.50)")
plt.show()

In [ ]:
coef = pd.Series(model.coef_[0], index=X.columns).sort_values()
coef.plot(kind="barh", color=["red" if c > 0 else "green" for c in coef])
plt.title("model emsallari")
plt.xlabel("emsal (scaled data)")
plt.show()

emsali musbet olan sutun artdiqca risk artir, menfi olan ise riski azaldir (meselen two_factor_enabled kimi qoruyucu sutunlar menfi cixa biler).

## 6. qerar: 0.30 ve 0.50 hedleri

In [ ]:
rows = []
for t in [0.30, 0.50]:
    p = (y_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    rows.append({"hed": t, "TP": tp, "FN": fn, "FP": fp, "TN": tn,
                 "accuracy": accuracy_score(y_test, p),
                 "precision": precision_score(y_test, p),
                 "recall": recall_score(y_test, p),
                 "f1": f1_score(y_test, p)})
pd.DataFrame(rows).round(3)

hed 0.30 olanda model daha asan "riskli" deyir, ona gore daha cox riskli giris tutulur (recall artir, FN azalir), amma FP de artir (precision asagi dusur). cedvelde bunu gorurem. tehlukesizlikde buraxilmis riskli giris daha bahalidir, ona gore **0.30 daha uygundur**, FP-nin artmasi ise qebul edilebilen xerc sayilir. (reqemleri yuxaridaki cedvelden yoxlayin)

### 3 yeni giris ucun proqnoz

In [ ]:
new_logins = pd.DataFrame([
    # sakit, adi giris
    {"login_hour": 10, "failed_attempts_24h": 0, "unknown_device": 0, "new_country": 0,
     "ip_risk_score": 15.0, "password_reset_7d": 0, "two_factor_enabled": 1,
     "account_age_days": 900, "last_login_gap_hours": 12.0, "vpn_used": 0},
    # gece, yeni olke, cox ugursuz cehd
    {"login_hour": 3, "failed_attempts_24h": 6, "unknown_device": 1, "new_country": 1,
     "ip_risk_score": 85.0, "password_reset_7d": 1, "two_factor_enabled": 0,
     "account_age_days": 40, "last_login_gap_hours": 200.0, "vpn_used": 1},
    # orta hal
    {"login_hour": 22, "failed_attempts_24h": 2, "unknown_device": 1, "new_country": 0,
     "ip_risk_score": 50.0, "password_reset_7d": 0, "two_factor_enabled": 1,
     "account_age_days": 300, "last_login_gap_hours": 60.0, "vpn_used": 1},
])

new_s = scaler.transform(imputer.transform(new_logins[X.columns]))
new_proba = model.predict_proba(new_s)[:, 1]

result = new_logins.copy()
result["risk_ehtimali"] = new_proba.round(3)
result["netice_0.30"] = np.where(new_proba >= 0.30, "riskli", "normal")
result["netice_0.50"] = np.where(new_proba >= 0.50, "riskli", "normal")
result[["risk_ehtimali", "netice_0.30", "netice_0.50"]]

## yekun
- melumat 1200 setir, login_id cixarildi, bosluqlar median ile dolduruldu
- imputer ve scaler yalniz train-e fit olundu
- logistic regression qurdum, confusion matrix ve metrikalari hesabladim
- FN (riskli girisi normal sanmaq) daha tehlukelidir, ona gore recall vacibdir
- hed 0.30 daha cox riskli giris tutur, FP bir az artir amma tehlukesizlik ucun deyer
- reqemleri notebook-u isledenden sonra cedvellerden yazmaq lazimdir